# Exploring Sample Data
Exploring all posts with >=1 comment over time period spanning January 1st, 2026 to November 1st, 2026.

In [1]:
import pandas as pd

In [6]:
sample_posts_path = "data/initial_eda/subreddit_resumes_posts.parquet"
posts = pd.read_parquet(sample_posts_path)

## Validating Data
Validating the data is what we expect by investigating distributions of values.

In [11]:
# validating the time period was respected
posts["created_utc"] = pd.to_datetime(posts["created_utc"], unit="s")
posts["created_utc"].describe()

count                         20827
mean     2026-05-14 14:14:26.002256
min             2026-01-01 00:29:21
25%      2026-02-25 15:13:13.500000
50%             2026-05-04 21:39:57
75%             2026-07-28 12:10:49
max             2026-10-31 22:55:34
Name: created_utc, dtype: object

In [16]:
posts["score"].describe()

count    20827.000000
mean         1.948384
std         14.389836
min          0.000000
25%          1.000000
50%          1.000000
75%          1.000000
max       1175.000000
Name: score, dtype: float64

In [32]:
# min comments is 1
posts["num_comments"].describe()

count    20827.000000
mean         2.733231
std          5.814029
min          1.000000
25%          1.000000
50%          1.000000
75%          3.000000
max        271.000000
Name: num_comments, dtype: float64

In [19]:
posts["id"].nunique()

20827

In [24]:
# for the subreddit r/resumes
posts["subreddit"].nunique()

1

In [27]:
posts["url"].value_counts()

url
                                                                                                      410
https://ibb.co/FkH9bW5S                                                                                 4
/r/CVwriting/comments/1vdv2du/final_year_psychology_student_rate_my_cv/                                 3
/r/EngineeringResumes/comments/1vayoet/2_yoe_international_ms_grad_resume_review/                       3
https://i.redd.it/tbz1om0o95gh1.png                                                                     3
                                                                                                     ... 
https://www.reddit.com/r/resumes/comments/1q47e05/1_yoe_qos_monitoring_and_drive_test_analyst_ran/      1
https://www.reddit.com/r/resumes/comments/1q47d2u/30_yoe_unemployed_supply_chainlogistics_united/       1
https://www.reddit.com/r/resumes/comments/1q47bei/1_yoe_grad_student_ran_engineer_kpi_performance/      1
https://i.redd.it/dexw9s8rhfbg1.png       

In [ ]:
posts["url"].value_counts()

selftext
[removed]                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  9071
                                                                                                               

In [31]:
posts["title"].value_counts()

title
Roast my resume                                                                                      84
[ Removed by moderator ]                                                                             34
Resume Review                                                                                        34
Resume help                                                                                          31
Review my resume                                                                                     28
                                                                                                     ..
[1 YoE, QoS Monitoring and Drive Test Analyst , RAN Performance Optimization Engineer , Tunisia ]     1
30 YoE, Unemployed, Supply Chain/Logistics, United States                                             1
[1 YoE, Grad Student/ RAN Engineer / KPI Performance & Optimization , Tunisia]                        1
Resume Review for Software Engineering - Business Hybrid  

## Filtering Data
We need to filter the data to ensure we only consider posts with relevant data. For relevant posts, the following must apply:
- Title must be of form **[X YoE, Current Role/Unemployed, Target Role, Country]**.
    - This is expected format according to r/resumes guidelines for submitting a resume review request.
- There must be text that was not removed, deleted, or otherwise unreadable.
- The url must contain a image of the resume (not anything else)


### Filter Text Column
- removed useless posts which contain the following: "removed", "deleted", or empty content.
- leaves 9658 of the original 20827 remaining (46.37% remaining).

In [42]:
text_len = posts["selftext"].str.len()

In [50]:
text_len_75, text_len_25 = text_len.quantile(0.75), text_len.quantile(0.25)
text_len_iqr = text_len_75 - text_len_25
text_len_lower, text_len_upper = text_len_25 - 1.5 * text_len_iqr, text_len_75 + 1.5 * text_len_iqr
text_requires_review = text_len <= text_len_upper

In [53]:
invalid_words = ["[removed]", "[deleted]"]
is_text_valid = (posts["selftext"].str.len() > 0) & (posts["selftext"].isin(invalid_words) == False)

In [55]:
is_text_valid.value_counts()

selftext
False    11169
True      9658
Name: count, dtype: int64

In [56]:
is_text_valid.shape

(20827,)

In [57]:
is_text_valid.value_counts() / is_text_valid.shape[0]

selftext
False    0.536275
True     0.463725
Name: count, dtype: float64

### Filter URL Column
- only included posts with ending "jpeg" and "png"
- leaves 9215 of the original 20827 remaining (44.24%).

In [63]:
url_endings = posts["url"].str.split(".").str[-1]

In [77]:
url_ending_len = url_endings.str.len()
url_ending_len_75, url_ending_len_25 = url_ending_len.quantile(0.75), url_ending_len.quantile(0.25)
url_ending_len_iqr = url_ending_len_75 - url_ending_len_25
url_ending_lower, url_ending_upper = url_ending_len_25 - 1.5 * url_ending_len_iqr, url_ending_len_75 + 1.5 * url_ending_len_iqr
is_likely_url = url_ending_len < url_ending_len_75

In [86]:
valid_endings = ["png", "jpeg"]
url_ending = posts["url"].str.split(".").str[-1]
is_url_valid = url_ending.isin(valid_endings) == True

In [87]:
is_url_valid.value_counts()

url
False    11612
True      9215
Name: count, dtype: int64

In [88]:
is_url_valid.value_counts() / is_url_valid.shape[0]

url
False    0.557545
True     0.442455
Name: count, dtype: float64

### Filter Title Column
- we were able to find the relevant pattern (**[X YoE, Current Role/Unemployed, Target Role, Country]**) in 8512 of the 20827 rows (40.87%).

In [90]:
posts["title"]

0        [1 YoE, Account Executive/Customer Success Hyb...
1        [1-2 YOE, Account Executive/Customer Success H...
2        I spent months building a complete job search ...
3        [4 YoE, Sub Teacher, Remote/USA, Looking to Pi...
4        [3 YoE, Art Director/Laid Off, 2D artist/conce...
                               ...                        
20822    [1 YoE, QoS Monitoring and Drive Test Analyst ...
20823    30 YoE, Unemployed, Supply Chain/Logistics, Un...
20824    [1 YoE, Grad Student/ RAN Engineer / KPI Perfo...
20825    Resume Review for Software Engineering - Busin...
20826    [1 YoE, IT Business Analyst Intern, Associate/...
Name: title, Length: 20827, dtype: str

In [218]:
# [X YoE, Current Role/Unemployed, Target Role, Country]
relevant_title_regex = r"\[([^,]+),\s*([^,]+),\s*([^,]+),\s*([^\]]+)\]"
parsed_title_parts = posts["title"].str.extract(relevant_title_regex).dropna()
parsed_title_parts.columns = ["yoe", "current_role", "target_role", "country"]
parsed_title_parts.head(2)

,yoe,current_role,target_role,country
0,1 YoE,Account Executive/Customer Success Hybrid,Customer Success,Toronto
3,4 YoE,Sub Teacher,Remote/USA,Looking to Pivot into Customer Service/IT Jobs


In [221]:
parsed_title_parts.shape

(8512, 4)

In [222]:
parsed_title_parts.shape[0] / posts.shape[0]

0.40870024487444184

#### Filter Years of Experience
- 7129 of 8512 (83.75%) remain after filtering out unparsable years of experience.

In [198]:
# find outliers for yoe
yoe_len = parsed_title_parts["yoe"].str.len()
yoe_iqr = yoe_len.quantile(0.75) - yoe_len.quantile(0.25)
yoe_lower, yoe_upper = yoe_len.quantile(0.25) - 1.5 * yoe_iqr, yoe_len.quantile(0.75) + 1.5 * yoe_iqr
yoe_needs_review = (yoe_len <= yoe_lower) | (yoe_len >= yoe_upper)
yoe_valid = ~yoe_needs_review
contains_yoe = (
    parsed_title_parts["yoe"].str.lower().str.contains("yoe") | 
    parsed_title_parts["yoe"].str.lower().str.contains("years") |
    parsed_title_parts["yoe"].str.lower().str.contains("yrs") |
    parsed_title_parts["yoe"].str.lower().str.contains("yr") |
    parsed_title_parts["yoe"].str.lower().str.contains("year")
)

In [199]:
yoe_lower

np.float64(3.5)

In [200]:
yoe_upper

np.float64(7.5)

In [201]:
parsed_title_parts["yoe"][yoe_needs_review].head(5)

73                 1.5 Years
92          4 Months (co-op)
121    4 Years of Experience
283           3.5 YoE + 2YoE
327         3rd Year student
Name: yoe, dtype: str

In [202]:
# 8076 of the 20827
parsed_title_parts["yoe"][yoe_valid].dropna().shape

(8076,)

In [203]:
contains_yoe.value_counts()

yoe
True     8317
False     195
Name: count, dtype: int64

In [204]:
(yoe_valid & contains_yoe).value_counts()

yoe
True     8028
False     484
Name: count, dtype: int64

In [205]:
# all mispelled or wrong
parsed_title_parts["yoe"][yoe_valid & ~contains_yoe].dropna()

156        4 YEO
579         9MOE
636      0 TEARS
932       Junior
1425      Resume
1482       7+ yo
1620     Student
2615       1 YeO
3454      Hiring
3492      10 MoE
3573        0Y0E
3716       0 YEO
3717       0 YEO
3725         23M
3851       3 YeO
3853       3 YeO
4663       0 yE0
6611       8 MoE
6936     Student
7334     1 month
8134        0 YE
8155        3moE
8269     6 month
8538       O YEO
8986       0 exp
9372       0 XoE
9532       2 YOW
9926     14+ EoE
10274    Fresher
10686       0 Ye
12402    Fresher
13269     15 yeo
13963     Senior
14812    3.5 YEO
14817    Fresher
15079     3 YoUE
15392    Student
15462      4 yes
16445     24 Y/O
16877      0 EoY
17208      7 MOE
17737    9 Y.O.E
17989    0 Y.O.E
18377    Student
19226      0 exp
19740       10YE
20290      5 EoY
20654        0YE
Name: yoe, dtype: str

In [206]:
parsed_title_parts["yoe"][yoe_valid].dropna().shape

(8076,)

In [207]:
yoe_upper

np.float64(7.5)

In [208]:
yoe_lower

np.float64(3.5)

In [223]:
yoe_len = parsed_title_parts["yoe"].str.len()
contains_valid_yoe_phrase = (
    parsed_title_parts["yoe"].str.lower().str.contains("yoe") | 
    parsed_title_parts["yoe"].str.lower().str.contains("years") |
    parsed_title_parts["yoe"].str.lower().str.contains("yrs") |
    parsed_title_parts["yoe"].str.lower().str.contains("yr") |
    parsed_title_parts["yoe"].str.lower().str.contains("year")
)
is_sufficient_yoe_len = (yoe_len > 3.5) & (yoe_len < 7)
is_yoe_valid = is_sufficient_yoe_len & contains_valid_yoe_phrase

In [224]:
is_yoe_valid.value_counts()

yoe
True     7129
False    1383
Name: count, dtype: int64

In [226]:
is_yoe_valid.value_counts()

yoe
True     7129
False    1383
Name: count, dtype: int64

In [227]:
is_yoe_valid.value_counts() / is_yoe_valid.shape[0]

yoe
True     0.837523
False    0.162477
Name: count, dtype: float64

#### Filter Country
- 8271 of 8512 (97.16%) remain after filtering out unparsable countries

In [234]:
country = parsed_title_parts["country"]

In [242]:
country_len = parsed_title_parts["country"].str.strip().str.len()
country_len.describe()

count    8512.000000
mean        8.985197
std         7.700511
min         0.000000
25%         4.000000
50%         7.000000
75%        13.000000
max       129.000000
Name: country, dtype: float64

In [243]:
is_not_empty_country = country_len > 0

In [244]:
is_not_empty_country.value_counts()

country
True     8511
False       1
Name: count, dtype: int64

In [252]:
country_len_iqr = country_len[is_not_empty_country].quantile(0.75) - country_len[is_not_empty_country].quantile(0.25)
country_len_lower, country_len_upper = country_len[is_not_empty_country].quantile(0.25) - 1.5 * country_len_iqr, country_len[is_not_empty_country].quantile(0.75) + 1.5 * country_len_iqr
country_len_lower

np.float64(-9.5)

In [253]:
country_len_upper

np.float64(26.5)

In [256]:
country[is_not_empty_country & (country_len > 26)]

3           Looking to Pivot into Customer Service/IT Jobs
15                       Looking to Relocate to Boston, MA
105                            Open to Relocation Anywhere
176      PMO Analyst, Project Coordinator, Implementati...
178      Aiming for an engineering research opportunity...
                               ...                        
19950                      Finance/Operations role, Canada
20161                          Finance, Legal) or Tech, UK
20263    Digital creative, Graphic designer, interdisci...
20380            risk and compliance desire, United States
20415             Product/UX Research-related, USA Citizen
Name: country, Length: 223, dtype: str

In [257]:
country[is_not_empty_country & (country_len < 26)]

0              Toronto
4           Remote/USA
5           Remote/USA
6        Remote/USA/EU
7        United States
             ...      
20813           Canada
20817              USA
20821              USA
20822         Tunisia 
20826    United States
Name: country, Length: 8270, dtype: str

In [258]:
is_valid_country = (parsed_title_parts["country"].str.len() > 0) & (parsed_title_parts["country"].str.len() < 26)

In [259]:
is_valid_country.value_counts()

country
True     8271
False     241
Name: count, dtype: int64

In [260]:
is_valid_country.value_counts() / is_valid_country.shape[0]

country
True     0.971687
False    0.028313
Name: count, dtype: float64

#### Parsing Current Role & Target Role